In [16]:
import glob
import os
import json
import pandas as pd
import glob
import os
import json
import pandas as pd
import glob
import os
from neal import SimulatedAnnealingSampler
import json
import networkx as nx
import numpy as np
import time
import dimod
import glob
import os
import json


from sidion_qubo import load_data, get_G_ising, solve_ising, export_results_to_json, Ising

import cvxpy as cvx
from qbo_sdp import qbo_sdp_01, goemans_williamson_loop



def solve_GW(Q_offset):
    Q, offset = Q_offset
    Q *= 28
    
    (value, (x, X, Y, status, runtime)) = qbo_sdp_01(
        Q, solver=cvx.MOSEK, 
        linear_constraint=None, 
        quadratic_constraints=None, verbose=True)
       
    gw_x, gw_info = goemans_williamson_loop(
        Y, Q,
        linear_constraint=None,
        quadratic_constraints=None,
        tol_lin=1e-6, tol_quad=1e-6,
        max_trials=10, seed=42,
        local_improve=None,               # or "hill-climber", function for local improvement, TBD
        stop_at_first_feasible=False)

    return {'value': gw_info["ub_value"]/28+offset, 'sdp': value/28+offset}


def ising2qubo(ising):
    Q, nodes = nx.attr_matrix(ising, edge_attr='J')
    Q = np.triu(Q, 1)
    node_h = nx.get_node_attributes(ising, 'h')
    L = np.array([node_h[node] for node in nodes])
    
    QUBO = 4*Q
    
    offset = -L.sum() + Q.sum()
    np.fill_diagonal(QUBO, 2*L - 2*Q.sum(axis=0) - 2*Q.sum(axis=1))

    return QUBO, offset

def solve_qubo(Q_offset, sampler, **kwargs):
    Q, offset = Q_offset
    
    time_b = time.time()
    bqm = dimod.BQM.from_qubo(Q, offset=offset)
    sampleset = sampler.sample(bqm, **kwargs)
    running_time = time.time() - time_b
    
    return {'value': sampleset.first.energy, 'X': sampleset.first.sample, 'sampler_response': sampleset, 'ising': ising, 
            'time': running_time,
            'sampler': {'class': type(sampler).__name__, 'kwargs': kwargs}}




In [27]:
solver_type = 'Advantage2_system1.8'
case_path = '../data/{}/send_{}/m{}/case{}/beta{}/'
list_of_cases = sorted(glob.glob(case_path.format(solver_type, '8', '*', '*', 0.0)))
list_of_cases = sorted(glob.glob(case_path.format(solver_type, '8', '1', '*', 0.0)))
#isings = ['J_log', 'J_qac']
isings = ['J_log']
num_reads = [10]

list_of_cases

['../data/Advantage2_system1.8/send_8/m1/case0/beta0.0/',
 '../data/Advantage2_system1.8/send_8/m1/case1/beta0.0/',
 '../data/Advantage2_system1.8/send_8/m1/case2/beta0.0/',
 '../data/Advantage2_system1.8/send_8/m1/case3/beta0.0/',
 '../data/Advantage2_system1.8/send_8/m1/case4/beta0.0/',
 '../data/Advantage2_system1.8/send_8/m1/case5/beta0.0/',
 '../data/Advantage2_system1.8/send_8/m1/case6/beta0.0/',
 '../data/Advantage2_system1.8/send_8/m1/case7/beta0.0/',
 '../data/Advantage2_system1.8/send_8/m1/case8/beta0.0/',
 '../data/Advantage2_system1.8/send_8/m1/case9/beta0.0/']

In [28]:
for ising_type in isings:
    for case in list_of_cases:
        with open(f'{case}{ising_type}.json') as f:
            data = json.load(f)

        ising = Ising().deserialize(data)

        for n in num_reads:
            print(f'Solving {case} with GW ...')

            
            result = solve_GW(ising2qubo(ising))
            result['case'] = case
            print(' ... Done.')

            
            output_file_folder = case.replace(f'data/{solver_type}', f'results/{solver_type}/GW2')
            os.makedirs(os.path.dirname(output_file_folder), exist_ok=True)
            output_file_name = f"{output_file_folder}GW_{ising_type}.json"

            print(output_file_name)
            
            with open(output_file_name, 'w') as f:
                json.dump(result, f)
            #print(f'Exporting results to {output_file_name} ...')
            #export_results_to_json(result, output_file_name)
            print(' ... Done.')

Solving ../data/Advantage2_system1.8/send_8/m1/case0/beta0.0/ with GW ...
                                     CVXPY                                     
                                     v1.6.5                                    
(CVXPY) Dec 09 12:01:13 PM: Your problem has 169 variables, 206 constraints, and 0 parameters.
(CVXPY) Dec 09 12:01:13 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Dec 09 12:01:13 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Dec 09 12:01:13 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Dec 09 12:01:13 PM: Your problem is compiled with the CPP canonicalization backend.
-------------------------------------------------------------------------------
                                  Compilation                                  
---------------------------------------------------------------------------